# Harry Potter GPT — full Colab pipeline (Pretrain -> SFT -> HF convert -> DPO)

Re-runs the whole thing end to end on one Colab GPU, since the previous SFT/DPO weights were lost.

**Key difference from the old Kaggle notebook: every checkpoint is written straight to Google Drive as training happens**, not zipped and uploaded after the fact. If the Colab runtime disconnects mid-run, nothing is lost — just re-run the corresponding cell, `init_from='resume'` picks it back up.

Runtime: Runtime -> Change runtime type -> T4 GPU (free tier is fine, this is a 124M model).

Run the cells top to bottom. Each stage prints samples so you can eyeball progress before moving to the next one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/harry-potter-gpt'
for d in ['out-harry-potter', 'harry-potter-hf', 'harry-potter-hf-dpo', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
print('Drive folders ready at', DRIVE_ROOT)

In [ ]:
!git clone https://github.com/abhijitdalal26/harry-potter-gpt.git /content/harry-potter-gpt
%cd /content/harry-potter-gpt/nanoGPT

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q tiktoken transformers datasets trl accelerate

## Stage 0 — baseline (untouched pretrained GPT-2, no HP training at all)
Sanity check + the "before" reference point for the final comparison.

In [ ]:
!python sample.py --init_from=gpt2 --start="Harry Potter" --num_samples=2 --max_new_tokens=150

## Data prep — tokenize the 7 HP books with the GPT-2 BPE tokenizer
The raw `harry_potter.txt` is already in the repo; the tokenized `.bin` files are gitignored (large + trivial to regenerate).

In [ ]:
!python data/harry_potter/prepare.py

## Stage 1 — Continued pretraining on the HP books
Starts from pretrained GPT-2 (124M), keeps training on the 7 books so it picks up world/characters/style. Checkpoint is written straight to Drive.

In [ ]:
!python train.py config/finetune_harry_potter.py --out_dir=/content/drive/MyDrive/harry-potter-gpt/out-harry-potter --compile=False

In [ ]:
!python sample.py --out_dir=/content/drive/MyDrive/harry-potter-gpt/out-harry-potter --start="Harry Potter" --num_samples=3 --max_new_tokens=200

## Stage 2 — Supervised fine-tuning (SFT) on the chat format
`hp_sft_data.txt` (4,321 lines, `<|user|>`/`<|assistant|>` format) is already in the repo. This resumes from the Stage 1 checkpoint in Drive and continues training in the same directory, with loss masked to only the assistant tokens.

In [ ]:
!python data/harry_potter_sft/prepare_sft.py

In [ ]:
!python train.py config/harry_potter_sft.py --out_dir=/content/drive/MyDrive/harry-potter-gpt/out-harry-potter --compile=False

In [ ]:
USER = chr(60) + "|user|" + chr(62)
ASST = chr(60) + "|assistant|" + chr(62)
prompt = f"{USER} Who is Harry Potter?\n{ASST}"
!python sample.py --out_dir=/content/drive/MyDrive/harry-potter-gpt/out-harry-potter --start="$prompt" --num_samples=3 --max_new_tokens=200

## Stage 3 — Convert the nanoGPT checkpoint to HuggingFace format
Needed so TRL's `DPOTrainer` can use it. The one thing to get right: nanoGPT stores `attn`/`mlp` weights as `Conv1D` (transposed) vs HuggingFace's `nn.Linear` — those four weight matrices get transposed on the way across, everything else copies straight.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Config, GPT2Tokenizer

CKPT_PATH  = '/content/drive/MyDrive/harry-potter-gpt/out-harry-potter/ckpt.pt'
OUTPUT_DIR = '/content/drive/MyDrive/harry-potter-gpt/harry-potter-hf'

print("Loading checkpoint...")
ckpt = torch.load(CKPT_PATH, map_location='cpu')
model_args = ckpt['model_args']
state_dict = ckpt['model']
print("Model args:", model_args)

unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

key_mapping = {
    'transformer.wte.weight': 'transformer.wte.weight',
    'transformer.wpe.weight': 'transformer.wpe.weight',
    'transformer.ln_f.weight': 'transformer.ln_f.weight',
    'transformer.ln_f.bias': 'transformer.ln_f.bias',
    'lm_head.weight': 'lm_head.weight',
}
for i in range(model_args['n_layer']):
    key_mapping.update({
        f'transformer.h.{i}.ln_1.weight': f'transformer.h.{i}.ln_1.weight',
        f'transformer.h.{i}.ln_1.bias': f'transformer.h.{i}.ln_1.bias',
        f'transformer.h.{i}.ln_2.weight': f'transformer.h.{i}.ln_2.weight',
        f'transformer.h.{i}.ln_2.bias': f'transformer.h.{i}.ln_2.bias',
        f'transformer.h.{i}.attn.c_attn.weight': f'transformer.h.{i}.attn.c_attn.weight',
        f'transformer.h.{i}.attn.c_attn.bias': f'transformer.h.{i}.attn.c_attn.bias',
        f'transformer.h.{i}.attn.c_proj.weight': f'transformer.h.{i}.attn.c_proj.weight',
        f'transformer.h.{i}.attn.c_proj.bias': f'transformer.h.{i}.attn.c_proj.bias',
        f'transformer.h.{i}.mlp.c_fc.weight': f'transformer.h.{i}.mlp.c_fc.weight',
        f'transformer.h.{i}.mlp.c_fc.bias': f'transformer.h.{i}.mlp.c_fc.bias',
        f'transformer.h.{i}.mlp.c_proj.weight': f'transformer.h.{i}.mlp.c_proj.weight',
        f'transformer.h.{i}.mlp.c_proj.bias': f'transformer.h.{i}.mlp.c_proj.bias',
    })

transposed_keys = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

new_state_dict = {}
for nano_key, hf_key in key_mapping.items():
    if nano_key in state_dict:
        tensor = state_dict[nano_key]
        if any(nano_key.endswith(k) for k in transposed_keys):
            tensor = tensor.t()
        new_state_dict[hf_key] = tensor
    else:
        print(f"WARNING: missing key {nano_key}")

hf_config = GPT2Config(
    vocab_size=model_args['vocab_size'],
    n_positions=model_args['block_size'],
    n_embd=model_args['n_embd'],
    n_layer=model_args['n_layer'],
    n_head=model_args['n_head'],
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, use_cache=True,
)

hf_model = GPT2LMHeadModel(hf_config)
hf_model.load_state_dict(new_state_dict, strict=False)
hf_model.save_pretrained(OUTPUT_DIR)

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved HF model to", OUTPUT_DIR)

In [ ]:
# quick sanity check the conversion actually worked
model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).cuda().eval()
tok = GPT2Tokenizer.from_pretrained(OUTPUT_DIR)
prompt = f"{USER} Who is Harry Potter?\n{ASST}"
ids = tok(prompt, return_tensors='pt').to('cuda')
out = model.generate(**ids, max_new_tokens=150, do_sample=True, temperature=0.8, top_p=0.9, top_k=50, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=False))

## Stage 4 — DPO (preference alignment)
`dpo_data.json` (347 chosen/rejected pairs) is already in the repo. `DPOTrainer` trains the SFT model to prefer the fan-toned response over the flat/generic one, directly, no separate reward model.

In [ ]:
!python data/harry_potter_dpo/prepare_dpo.py

In [ ]:
%%writefile train_dpo_colab.py
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_from_disk
from trl import DPOTrainer, DPOConfig

MODEL_DIR = "/content/drive/MyDrive/harry-potter-gpt/harry-potter-hf"
DPO_DATA_DIR = "data/harry_potter_dpo"
OUTPUT_DIR = "/content/drive/MyDrive/harry-potter-gpt/harry-potter-hf-dpo"

model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

ref_model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)

train_dataset = load_from_disk(f"{DPO_DATA_DIR}/train")
eval_dataset = load_from_disk(f"{DPO_DATA_DIR}/val")
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    max_length=384,
    max_prompt_length=192,
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=True,
    precompute_ref_log_probs=True,
)

trainer = DPOTrainer(
    model=model, ref_model=ref_model, args=dpo_config,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("DPO model saved to", OUTPUT_DIR)

In [ ]:
# NOTE: if this errors on DPOConfig/DPOTrainer kwargs, TRL's API has likely drifted
# since this was written (mid-2026) — check `python -c "import trl; print(trl.__version__)"`
# and drop/rename whichever kwarg it complains about.
!python train_dpo_colab.py

## Final showcase — Base vs SFT vs DPO, basic -> advanced questions
Same question set run against all three stages. Prints here AND saves to a markdown file in Drive, so the transcript survives even if this Colab session is closed without copying anything by hand.

In [ ]:
import torch, os
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from model import GPTConfig, GPT
import tiktoken

DRIVE_ROOT = '/content/drive/MyDrive/harry-potter-gpt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# base (nanoGPT format, stage 1 pretrain-only checkpoint)
ckpt = torch.load(f'{DRIVE_ROOT}/out-harry-potter/ckpt.pt', map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
base_model = GPT(gptconf)
sd = ckpt['model']
for k in list(sd.keys()):
    if k.startswith('_orig_mod.'):
        sd[k[len('_orig_mod.'):]] = sd.pop(k)
base_model.load_state_dict(sd)
base_model.eval().to(device)
enc = tiktoken.get_encoding('gpt2')

# SFT + DPO (HF format)
tok = GPT2Tokenizer.from_pretrained(f'{DRIVE_ROOT}/harry-potter-hf')
tok.pad_token = tok.eos_token
sft_model = GPT2LMHeadModel.from_pretrained(f'{DRIVE_ROOT}/harry-potter-hf').to(device).eval()
dpo_model = GPT2LMHeadModel.from_pretrained(f'{DRIVE_ROOT}/harry-potter-hf-dpo').to(device).eval()

USER = chr(60) + "|user|" + chr(62)
ASST = chr(60) + "|assistant|" + chr(62)

# basic -> advanced, mirrors the DPO training data's own category spread
questions = [
    "Who is Harry Potter?",
    "What is the Mirror of Erised?",
    "Why did Voldemort fail to kill Harry as a baby?",
    "Is Dumbledore a good person?",
    "Does Snape try to save Harry Potter?",
    "Why didn't they just use Veritaserum on Sirius to prove he was innocent?",
    "I just finished Prisoner of Azkaban and I am destroyed about Sirius",
    "I have a theory that Dumbledore is actually Death from the Three Brothers tale",
]

def gen_base(prompt, max_new_tokens=150):
    ids = enc.encode(prompt, allowed_special={"<|endoftext|>"})
    x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]
    with torch.no_grad():
        y = base_model.generate(x, max_new_tokens, temperature=0.8, top_k=50)
    return enc.decode(y[0].tolist()[len(ids):]).strip()

def gen_hf(model, prompt, max_new_tokens=150):
    inp = tok(prompt, return_tensors='pt').to(device)
    n = inp['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=0.8, top_p=0.9, top_k=50, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][n:], skip_special_tokens=True).strip()

lines = ["# Harry Potter GPT — stage comparison transcript\n"]
for q in questions:
    prompt = f"{USER} {q}\n{ASST}"
    base_ans = gen_base(prompt)
    sft_ans = gen_hf(sft_model, prompt)
    dpo_ans = gen_hf(dpo_model, prompt)
    block = f"\n## Q: {q}\n\n**Base (pretrain only):** {base_ans}\n\n**SFT:** {sft_ans}\n\n**DPO:** {dpo_ans}\n"
    print(block)
    lines.append(block)

with open(f'{DRIVE_ROOT}/results/stage_comparison.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print("\n\nSaved full transcript to", f'{DRIVE_ROOT}/results/stage_comparison.md')

## Done

Everything of value now lives in Google Drive under `harry-potter-gpt/`:
- `out-harry-potter/ckpt.pt` — Stage 1 nanoGPT checkpoint
- `harry-potter-hf/` — Stage 2 SFT model (HuggingFace format)
- `harry-potter-hf-dpo/` — Stage 4 DPO model (HuggingFace format)
- `results/stage_comparison.md` — the basic-to-advanced transcript across all three stages

Next: download `results/stage_comparison.md` (and, if you want to host it, the `harry-potter-hf-dpo/` folder) and share it back — that's what turns into the real conversation examples on the portfolio page, and what a hosted demo would run off of.